# 1.1 — tagged / untagged SPNs from the exported tables (server side)

Self-contained: needs only `numpy` and `pandas`, no optonpx / SpikeInterface, so it
can be copied to the lab server and run next to the uploaded files.

Inputs (one pair per session, renamed on upload to `<animal>_<session>_...`):

| file | from | content |
|---|---|---|
| `<animal>_<session>_optotagging_summary.csv` | notebook 3.2 | one row per unit × laser color × power: SALT p, reliability, n responded, waveform correlation, ... |
| `<animal>_<session>_unit_classification.csv` | notebook 3.3 | one row per unit: raw-waveform features (peak width, peak-to-valley) and behavior-session firing rate |

All decisions are **re-made here from the metric columns** with the thresholds in the
parameter cell; the decision columns already in the files (`is_tagged`,
`color_tagged`, `blue_tagged`, `red_tagged`, `is_spn`, ...) are only used for a
consistency check against the local run.

Rules (same as notebooks 3.2 and 3.3):

- **condition passes** (one unit × color × power):
  `salt_p < salt_alpha` and `reliability >= min_reliability` and
  `n_responded >= min_responded` and (if `require_wave_corr`) `wave_corr >= min_wave_corr`
  (NaN fails everything).
- **color tagged**: the unit passes at any power of that color.
- **SPN**: waveform features did not fail and `peak_width_us > min_peak_width_us` and
  `p2v_us > min_p2v_us` and `firing_rate_hz < max_fr_hz` (strict, as in Dan's MATLAB).

`unit_id` is the Kilosort/Phy cluster id = `original_cluster_id` in the NWB units
table (= the NWB unit id for these exports). Only units that were tested in 4.2 are
in the files (Phy `good`); every listed unit was tested, so "untagged" means
*tested and did not pass*, not "never tested".

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd

# ======== parameters to change ========
data_dir = Path(".")              # folder holding the uploaded CSVs
animal   = "IM-1979"
session  = "2026-09-04"

# optotagging (notebook 3.2 defaults)
OPTO = {
    "salt_alpha":        0.001,   # salt_p <  this
    "min_reliability":   0.25,    # reliability >= this
    "min_responded":     10,      # n_responded >= this
    "require_wave_corr": True,
    "min_wave_corr":     0.9,     # wave_corr >= this
}
# putative SPN (notebook 3.3 defaults; strict inequalities)
SPN = {
    "min_peak_width_us": 90.0,    # peak_width_us >  this
    "min_p2v_us":        560.0,   # p2v_us        >  this
    "max_fr_hz":         15.0,    # firing_rate_hz < this
}
TAG_COLORS = None                 # None = every color in the file, or e.g. ["blue_laser"]
save_output = False               # write <animal>_<session>_tagged_units.csv into data_dir
# ======================================

prefix          = f"{animal}_{session}"
summary_path    = data_dir / f"{prefix}_optotagging_summary.csv"
classified_path = data_dir / f"{prefix}_unit_classification.csv"
for p in (summary_path, classified_path):
    if not p.is_file():
        raise FileNotFoundError(p)
print(f"summary         = {summary_path}")
print(f"classification  = {classified_path}")

In [ ]:
# --- load and sanity-check the two tables ---
summary    = pd.read_csv(summary_path)
classified = pd.read_csv(classified_path)

need_summary = ["unit_id", "color", "power", "salt_p", "reliability", "n_responded", "wave_corr"]
need_class   = ["unit_id", "feature_failed", "peak_width_us", "p2v_us", "firing_rate_hz"]
for name, df, need in (("summary", summary, need_summary), ("classification", classified, need_class)):
    missing = [c for c in need if c not in df.columns]
    if missing:
        raise KeyError(f"{name} is missing columns {missing}")

if classified["unit_id"].duplicated().any():
    raise ValueError("unit_classification has duplicated unit_id rows")
if summary.duplicated(["unit_id", "color", "power"]).any():
    raise ValueError("optotagging_summary has duplicated (unit_id, color, power) rows")

only_summary = sorted(set(summary["unit_id"]) - set(classified["unit_id"]))
only_class   = sorted(set(classified["unit_id"]) - set(summary["unit_id"]))
if only_summary or only_class:
    print(f"WARNING units only in summary: {only_summary}\n        units only in classification: {only_class}")

colors = sorted(summary["color"].unique()) if TAG_COLORS is None else list(TAG_COLORS)
print(f"{classified['unit_id'].nunique()} units | {len(summary)} opto rows | "
      f"colors {sorted(summary['color'].unique())} | powers {list(summary['power'].unique())}")
print(f"colors used for tagging: {colors}")

In [ ]:
# --- optotagging: per (unit, color, power) condition, then per (unit, color) ---
cond = summary[["unit_id", "color", "power", "salt_p", "reliability", "n_responded", "wave_corr"]].copy()
cond["salt_ok"]     = cond["salt_p"] < OPTO["salt_alpha"]
cond["response_ok"] = ((cond["reliability"] >= OPTO["min_reliability"])
                       & (cond["n_responded"] >= OPTO["min_responded"]))
cond["wave_ok"]     = (cond["wave_corr"] >= OPTO["min_wave_corr"]) if OPTO["require_wave_corr"] else True
cond["passes"]      = cond["salt_ok"] & cond["response_ok"] & cond["wave_ok"]   # NaN compares False

cond_used = cond[cond["color"].isin(colors)]
color_tagged = (cond_used.groupby(["unit_id", "color"])["passes"].any()
                .unstack("color").reindex(columns=colors).fillna(False).astype(bool))
color_tagged.columns = [f"{c.removesuffix('_laser')}_tagged" for c in color_tagged.columns]

# strongest evidence per unit: the passing condition with the lowest SALT p
best = (cond_used[cond_used["passes"]].sort_values("salt_p")
        .groupby("unit_id").head(1).set_index("unit_id")[["color", "power", "salt_p", "reliability", "wave_corr"]]
        .add_prefix("best_"))

print("passing conditions per color x power:")
print(cond.pivot_table(index="color", columns="power", values="passes", aggfunc="sum", sort=False))

In [ ]:
# --- putative SPN from the raw-waveform features and firing rate ---
spn = classified[["unit_id", "feature_failed", "peak_width_us", "p2v_us", "firing_rate_hz"]].copy()
feature_ok = ~spn["feature_failed"].astype(bool)
width_ok   = spn["peak_width_us"]  > SPN["min_peak_width_us"]
p2v_ok     = spn["p2v_us"]         > SPN["min_p2v_us"]
fr_ok      = spn["firing_rate_hz"] < SPN["max_fr_hz"]
spn["is_spn"] = feature_ok & width_ok & p2v_ok & fr_ok

# first failing criterion, in the same order as notebook 3.3
spn["spn_fail_reason"] = np.select(
    [~feature_ok, ~width_ok, ~p2v_ok, ~fr_ok],
    ["waveform feature failed", "peak width", "peak-to-valley", "firing rate"], default="")
print(spn["spn_fail_reason"].replace("", "(SPN)").value_counts().to_string())

In [ ]:
# --- combine: one row per unit ---
units = (spn.set_index("unit_id")
         .join(color_tagged, how="left")
         .join(best, how="left"))
tag_cols = list(color_tagged.columns)
units[tag_cols] = units[tag_cols].fillna(False).astype(bool)
units["any_tagged"] = units[tag_cols].any(axis=1)
units["group"] = np.select(
    [units["is_spn"] & units["any_tagged"], units["is_spn"] & ~units["any_tagged"],
     ~units["is_spn"] & units["any_tagged"]],
    ["tagged_spn", "untagged_spn", "tagged_non_spn"], default="untagged_non_spn")
units = units.sort_index()

tagged_spn   = units.index[units["group"] == "tagged_spn"].tolist()
untagged_spn = units.index[units["group"] == "untagged_spn"].tolist()

print(units["group"].value_counts().reindex(
    ["tagged_spn", "untagged_spn", "tagged_non_spn", "untagged_non_spn"], fill_value=0).to_string())
print({c: int(units[c].sum()) for c in tag_cols})
print(f"\ntagged SPN   ({len(tagged_spn)}): {tagged_spn}")
print(f"\nuntagged SPN ({len(untagged_spn)}): {untagged_spn}")

In [ ]:
# --- consistency check against the decisions made locally (4.2 / 4.4) ---
# With the default thresholds every line should say 0 differences; after changing
# thresholds, differences are expected and just show what moved.
checks = {"is_spn": "is_spn"}
checks.update({c: c for c in tag_cols if c in classified.columns})
local = classified.set_index("unit_id")
for ours, theirs in checks.items():
    if theirs not in local.columns:
        continue
    diff = units.index[units[ours] != local[theirs].reindex(units.index).astype(bool)].tolist()
    print(f"{ours:<14} differs from the local file for {len(diff)} unit(s){': ' + str(diff) if diff else ''}")

In [ ]:
# --- output table (unit_id = NWB original_cluster_id) ---
out_cols = ["group", "is_spn", "spn_fail_reason", *tag_cols, "any_tagged",
            "peak_width_us", "p2v_us", "firing_rate_hz",
            "best_color", "best_power", "best_salt_p", "best_reliability", "best_wave_corr"]
result = units[out_cols].reset_index()
display(result.head())

if save_output:
    out_path = data_dir / f"{prefix}_tagged_units.csv"
    result.to_csv(out_path, index=False)
    print(f"saved {out_path}")